In [ ]:
!pip install timm transformers wildlife-datasets wildlife-tools scikit-learn opencv-python-headless --quiet --upgrade
        

In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import random
import subprocess
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFile, ImageFilter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize

from huggingface_hub import login
from transformers import Sam3Model, Sam3Processor

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
        

In [ ]:
# =========================
# Config
# =========================

VERSION = "sam_arwildfusion_v3"
SEED = 20260424
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("GPU:", GPU_NAME)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORK_DIR / "animalclef_arwildfusion_v3"
VIEW_DIR = OUT_DIR / "views"
CACHE_DIR = OUT_DIR / "cache"
CKPT_DIR = OUT_DIR / "checkpoints"
REPORT_DIR = OUT_DIR / "reports"
SUB_DIR = OUT_DIR / "submissions"
for d in [OUT_DIR, VIEW_DIR, CACHE_DIR, CKPT_DIR, REPORT_DIR, SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BACKGROUND_COLOR = np.array([238, 238, 232], dtype=np.uint8)

if "L4" in GPU_NAME:
    SAM3_BATCH_SIZE = 12
    SAM3_MAX_SIDE = 1024
    TRAIN_P = 8
    TRAIN_K = 4
    TRAIN_NUM_WORKERS = 3
    INFER_BATCH = {"arwild": 96, "mega": 72, "miew": 96}
elif "P100" in GPU_NAME:
    SAM3_BATCH_SIZE = 8
    SAM3_MAX_SIDE = 896
    TRAIN_P = 8
    TRAIN_K = 4
    TRAIN_NUM_WORKERS = 2
    INFER_BATCH = {"arwild": 56, "mega": 40, "miew": 56}
else:
    SAM3_BATCH_SIZE = 8
    SAM3_MAX_SIDE = 896
    TRAIN_P = 8
    TRAIN_K = 4
    TRAIN_NUM_WORKERS = 3
    INFER_BATCH = {"arwild": 64, "mega": 48, "miew": 64}

PERSISTENT_WORKERS = TRAIN_NUM_WORKERS > 0
TRAIN_BATCH = TRAIN_P * TRAIN_K
GRAD_ACCUM = 1
MAX_EPOCHS = 4
STEPS_PER_EPOCH = 260
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TRIPLET_WEIGHT = 0.30
LABEL_SMOOTHING = 0.10
TRAIN_IMAGE_SIZE = 384
ARWILD_BACKBONE = "convnext_small.fb_in22k_ft_in1k_384"
ARC_SCALE = 30.0
ARC_MARGIN = 0.30

TTA_VARIANTS = ["identity", "hflip"]

PROMPT = {
    "LynxID2025": "lynx animal",
    "SalamanderID2025": "salamander animal",
    "SeaTurtleID2022": "sea turtle head animal",
    "TexasHornedLizards": "horned lizard animal",
}
SAM3_THRESHOLD = {
    "LynxID2025": 0.35,
    "SalamanderID2025": 0.30,
    "SeaTurtleID2022": 0.35,
    "TexasHornedLizards": 0.30,
}
SAM3_MASK_THRESHOLD = 0.50
CROP_TARGET_OCCUPANCY = {
    "LynxID2025": 0.74,
    "SalamanderID2025": 0.72,
    "SeaTurtleID2022": 0.78,
    "TexasHornedLizards": 0.76,
}

TRAIN_VIEW_WEIGHTS = {
    "original": 0.18,
    "mask_full": 0.34,
    "loose": 0.48,
}

VIEW_WEIGHT_PRESETS = {
    "balanced": {"original": 0.20, "mask_full": 0.35, "loose": 0.45},
    "contextual": {"original": 0.28, "mask_full": 0.32, "loose": 0.40},
    "focused": {"original": 0.12, "mask_full": 0.30, "loose": 0.58},
}

MODEL_WEIGHT_PRESETS = {
    "trained_dominant": {"trained": 0.55, "dominant": 0.30, "support": 0.15},
    "trained_balanced": {"trained": 0.42, "dominant": 0.38, "support": 0.20},
    "foundation_dominant": {"trained": 0.28, "dominant": 0.50, "support": 0.22},
}

SPECIES_CONFIG = {
    "LynxID2025": {
        "dominant_fixed": "miew",
        "support_fixed": "mega",
        "threshold_grid": [0.35, 0.39, 0.43, 0.47, 0.51],
        "qe_grid": [3, 5, 7],
        "split_cap": 120,
        "split_threshold": 0.36,
        "split_internal_sim": 0.55,
        "candidate_order": [("contextual", "trained_dominant"), ("balanced", "trained_balanced"), ("focused", "foundation_dominant")],
        "fallback_threshold": 0.43,
    },
    "SalamanderID2025": {
        "dominant_fixed": "mega",
        "support_fixed": "miew",
        "threshold_grid": [0.10, 0.12, 0.14, 0.16, 0.18],
        "qe_grid": [1, 3, 5],
        "split_cap": 10,
        "split_threshold": 0.10,
        "split_internal_sim": 0.70,
        "candidate_order": [("focused", "trained_dominant"), ("balanced", "trained_balanced"), ("focused", "foundation_dominant")],
        "fallback_threshold": 0.14,
    },
    "SeaTurtleID2022": {
        "dominant_fixed": "mega",
        "support_fixed": "miew",
        "threshold_grid": [0.40, 0.44, 0.48, 0.52],
        "qe_grid": [5, 8, 11],
        "split_cap": 22,
        "split_threshold": 0.40,
        "split_internal_sim": 0.64,
        "candidate_order": [("focused", "trained_balanced"), ("balanced", "foundation_dominant"), ("contextual", "trained_dominant")],
        "fallback_threshold": 0.48,
    },
    "TexasHornedLizards": {
        "dominant_fixed": "miew",
        "support_fixed": "mega",
        "threshold_grid": [0.26, 0.29, 0.32, 0.35],
        "qe_grid": [3, 5, 7],
        "split_cap": 48,
        "split_threshold": 0.26,
        "split_internal_sim": 0.58,
        "candidate_order": [("contextual", "trained_dominant"), ("balanced", "trained_balanced"), ("focused", "foundation_dominant")],
        "fallback_threshold": 0.30,
    },
}

print("SAM3 batch:", SAM3_BATCH_SIZE, "train batch:", TRAIN_BATCH, "infer batch:", INFER_BATCH)
        

In [ ]:
# =========================
# Data discovery
# =========================

token = os.environ.get("HF_TOKEN")
if token is None:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        token = None
if token:
    login(token=token, add_to_git_credential=False)
    print("[hf] logged in")
else:
    print("[hf] HF_TOKEN not found; public models only")


def find_competition_root():
    candidates = [
        Path("/kaggle/input/animal-clef-2026"),
        Path("/kaggle/input/competitions/animal-clef-2026"),
        Path("/content/animal-clef-2026"),
        Path.cwd() / "animal-clef-2026",
    ]
    for root in candidates:
        if (root / "metadata.csv").exists() and (root / "sample_submission.csv").exists():
            return root
    for base in [Path("/kaggle/input"), Path("/content"), Path.cwd()]:
        if base.exists():
            for p in base.rglob("metadata.csv"):
                if (p.parent / "sample_submission.csv").exists():
                    return p.parent
    raise FileNotFoundError("Competition data root not found.")


DATA_ROOT = find_competition_root()
metadata = pd.read_csv(DATA_ROOT / "metadata.csv").reset_index(drop=True)
sample_submission = pd.read_csv(DATA_ROOT / "sample_submission.csv")
metadata["row_idx"] = np.arange(len(metadata))
metadata["abs_path"] = metadata["path"].map(lambda p: str((DATA_ROOT / str(p)).resolve()))
metadata["identity"] = metadata.get("identity").fillna("").astype(str)
metadata["has_identity"] = metadata["identity"].ne("")
metadata["species_group"] = metadata["dataset"].astype(str)
print("DATA_ROOT:", DATA_ROOT)
display(metadata.groupby(["dataset", "split"]).size())
        

In [ ]:
# =========================
# SAM3 view regeneration
# =========================

VIEW_MANIFEST = REPORT_DIR / f"view_manifest_{VERSION}.csv"
FULL_DIR = VIEW_DIR / "mask_full_canvas"
LOOSE_DIR = VIEW_DIR / "mask_loose_square"
for d in [FULL_DIR, LOOSE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def find_external_view_manifests():
    manifests = []
    direct = [
        WORK_DIR / "animalclef_sam3_views_cache" / "reports" / "view_manifest_sam3_all_species.csv",
        Path.cwd() / "animalclef_sam3_views_cache" / "reports" / "view_manifest_sam3_all_species.csv",
    ]
    for manifest in direct:
        if manifest.exists():
            manifests.append(manifest)
    for base in [Path("/kaggle/input"), WORK_DIR, Path.cwd()]:
        if base.exists():
            try:
                for p in base.rglob("view_manifest_sam3_all_species.csv"):
                    if p.exists():
                        manifests.append(p)
            except Exception:
                pass
    unique = []
    seen = set()
    for manifest in manifests:
        rp = str(manifest.resolve())
        if rp not in seen:
            seen.add(rp)
            unique.append(manifest)
    return unique


def load_external_view_manifest():
    best_df = None
    best_manifest = None
    best_root = None
    best_score = (-1, -1)
    for manifest in EXTERNAL_VIEW_MANIFESTS:
        try:
            df = pd.read_csv(manifest)
        except Exception:
            continue
        required = {"row_idx", "mask_full_path", "loose_path", "mask_ok"}
        if not required.issubset(df.columns):
            continue
        score = (int(df["mask_ok"].fillna(False).sum()), len(df))
        if score > best_score:
            best_df = df.copy()
            best_manifest = manifest
            best_root = manifest.parent.parent if manifest.parent.name == "reports" else manifest.parent
            best_score = score
    return best_df, best_manifest, best_root


def cleanup_memory(tag=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
        if tag:
            free_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
            print(f"[gpu] {tag} free={free_gb:.2f}GB")


def load_resized_rgb(path, max_side=SAM3_MAX_SIDE):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(1.0, max_side / max(w, h))
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return img


def choose_mask(masks, scores):
    if masks is None or len(masks) == 0:
        return None
    arr = masks.detach().cpu().numpy().astype(bool)
    scores_np = scores.detach().cpu().numpy() if scores is not None else np.ones(len(arr), dtype="float32")
    h, w = arr.shape[-2:]
    total = float(h * w)
    best = None
    best_rank = -1.0
    for mask, score in zip(arr, scores_np):
        area = float(mask.sum())
        frac = area / max(total, 1.0)
        if frac < 0.004 or frac > 0.95:
            continue
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        cx = (xs.mean() / max(w - 1, 1)) - 0.5
        cy = (ys.mean() / max(h - 1, 1)) - 0.5
        centrality = 1.0 - min(1.0, math.sqrt(cx * cx + cy * cy) / 0.72)
        rank = float(score) * 0.64 + min(frac / 0.40, 1.0) * 0.20 + centrality * 0.16
        if rank > best_rank:
            best_rank = rank
            best = mask
    return best


def pad_to_square(arr, fill=BACKGROUND_COLOR):
    h, w = arr.shape[:2]
    side = max(h, w)
    out = np.zeros((side, side, 3), dtype=np.uint8)
    out[:] = fill
    y0 = (side - h) // 2
    x0 = (side - w) // 2
    out[y0:y0 + h, x0:x0 + w] = arr
    return out


def save_regenerated_views(abs_path, species, image, mask):
    src = Path(abs_path)
    full_species_dir = FULL_DIR / species
    loose_species_dir = LOOSE_DIR / species
    full_species_dir.mkdir(parents=True, exist_ok=True)
    loose_species_dir.mkdir(parents=True, exist_ok=True)

    full_path = full_species_dir / f"{src.stem}_maskfull.jpg"
    loose_path = loose_species_dir / f"{src.stem}_loose.jpg"
    if full_path.exists() and loose_path.exists():
        return str(full_path), str(loose_path), True

    img = np.array(image)
    if mask is None or mask.sum() == 0:
        masked_full = img
        loose = pad_to_square(img)
        ok = False
    else:
        masked_full = np.where(mask[..., None], img, BACKGROUND_COLOR[None, None, :])
        ys, xs = np.where(mask)
        y0, y1 = ys.min(), ys.max()
        x0, x1 = xs.min(), xs.max()
        box_h = y1 - y0 + 1
        box_w = x1 - x0 + 1
        side = int(math.ceil(max(box_h, box_w) / math.sqrt(CROP_TARGET_OCCUPANCY[species])))
        side = max(side, max(box_h, box_w) + 18)
        cy = (y0 + y1) / 2.0
        cx = (x0 + x1) / 2.0
        side = int(min(max(side, 64), max(img.shape[0], img.shape[1]) * 1.35))
        x0c = int(round(cx - side / 2))
        y0c = int(round(cy - side / 2))
        x1c = x0c + side
        y1c = y0c + side
        px0 = max(0, -x0c)
        py0 = max(0, -y0c)
        px1 = max(0, x1c - img.shape[1])
        py1 = max(0, y1c - img.shape[0])
        crop = masked_full[max(0, y0c):min(img.shape[0], y1c), max(0, x0c):min(img.shape[1], x1c)]
        if px0 or py0 or px1 or py1:
            crop = cv2.copyMakeBorder(
                crop,
                py0,
                py1,
                px0,
                px1,
                borderType=cv2.BORDER_CONSTANT,
                value=BACKGROUND_COLOR.tolist(),
            )
        loose = pad_to_square(crop)
        ok = True

    cv2.imwrite(str(full_path), cv2.cvtColor(masked_full, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    cv2.imwrite(str(loose_path), cv2.cvtColor(loose, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
    return str(full_path), str(loose_path), ok


def resolve_external_view_path(path_value, export_root, species_name, original_path, view_kind):
    if isinstance(path_value, str) and Path(path_value).exists():
        return str(Path(path_value))
    stem = Path(original_path).stem
    if view_kind == "mask_full":
        candidate = Path(export_root) / "mask_full_canvas" / species_name / f"{stem}_maskfull.jpg"
    else:
        candidate = Path(export_root) / "mask_loose_square" / species_name / f"{stem}_loose.jpg"
    if candidate.exists():
        return str(candidate)
    return original_path


def segment_batch(paths, species, batch_size=SAM3_BATCH_SIZE):
    paths = [str(p) for p in paths]
    outputs = [None] * len(paths)
    pending_slots, pending_paths, pending_images = [], [], []
    for slot, path in enumerate(paths):
        stem = Path(path).stem
        full_path = FULL_DIR / species / f"{stem}_maskfull.jpg"
        loose_path = LOOSE_DIR / species / f"{stem}_loose.jpg"
        if full_path.exists() and loose_path.exists():
            outputs[slot] = (str(full_path), str(loose_path), True)
        else:
            pending_slots.append(slot)
            pending_paths.append(Path(path))
            pending_images.append(load_resized_rgb(path))

    if len(pending_paths) == 0:
        return outputs

    prompt = PROMPT[species]
    for start in tqdm(range(0, len(pending_paths), batch_size), desc=f"SAM3 {species}"):
        batch_slots = pending_slots[start:start + batch_size]
        batch_paths = pending_paths[start:start + batch_size]
        batch_images = pending_images[start:start + batch_size]
        masks = [None] * len(batch_images)
        try:
            inputs = sam_processor(
                images=batch_images,
                text=[prompt] * len(batch_images),
                return_tensors="pt",
            ).to(DEVICE)
            with torch.inference_mode():
                if torch.cuda.is_available():
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        outputs_raw = sam_model(**inputs)
                else:
                    outputs_raw = sam_model(**inputs)
            results = sam_processor.post_process_instance_segmentation(
                outputs_raw,
                threshold=SAM3_THRESHOLD[species],
                mask_threshold=SAM3_MASK_THRESHOLD,
                target_sizes=inputs.get("original_sizes").tolist(),
            )
            for k, result in enumerate(results):
                masks[k] = choose_mask(result.get("masks"), result.get("scores"))
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and batch_size > 1:
                cleanup_memory(f"sam3 oom {species}")
                retry = segment_batch([str(p) for p in batch_paths], species, batch_size=1)
                for slot, item in zip(batch_slots, retry):
                    outputs[slot] = item
                continue
            raise

        for slot, path, image, mask in zip(batch_slots, batch_paths, batch_images, masks):
            outputs[slot] = save_regenerated_views(path, species, image, mask)

        cleanup_memory(f"sam3 {species} batch")

    return outputs


EXTERNAL_VIEW_MANIFESTS = find_external_view_manifests()
EXTERNAL_VIEW_DF, EXTERNAL_VIEW_MANIFEST_PATH, EXTERNAL_VIEW_EXPORT_ROOT = load_external_view_manifest()
print("Detected external SAM view manifests:")
for p in EXTERNAL_VIEW_MANIFESTS:
    print(" -", p)

if VIEW_MANIFEST.exists():
    view_manifest = pd.read_csv(VIEW_MANIFEST)
    print("[views] loaded manifest", VIEW_MANIFEST)
elif EXTERNAL_VIEW_DF is not None:
    view_manifest = EXTERNAL_VIEW_DF.copy()
    print("[views] loaded external manifest", EXTERNAL_VIEW_MANIFEST_PATH)
    print("[views] external export root", EXTERNAL_VIEW_EXPORT_ROOT)
else:
    print("[sam3] loading model")
    sam_model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE).eval()
    sam_processor = Sam3Processor.from_pretrained("facebook/sam3")
    print("[sam3] ready")
    rows = []
    for species, g in metadata.groupby("dataset"):
        results = segment_batch(g["abs_path"].tolist(), species)
        ok_count = sum(bool(x[2]) for x in results)
        print(f"[sam3] {species}: ok {ok_count}/{len(results)}")
        for row_idx, item in zip(g["row_idx"].tolist(), results):
            full_path, loose_path, ok = item
            rows.append({
                "row_idx": int(row_idx),
                "mask_full_path": full_path,
                "loose_path": loose_path,
                "mask_ok": bool(ok),
            })
    view_manifest = pd.DataFrame(rows)
    view_manifest.to_csv(VIEW_MANIFEST, index=False)
    del sam_model
    cleanup_memory("after sam3")

metadata = metadata.merge(view_manifest, on="row_idx", how="left")
if EXTERNAL_VIEW_DF is not None and EXTERNAL_VIEW_EXPORT_ROOT is not None:
    metadata["mask_full_path"] = metadata.apply(
        lambda row: resolve_external_view_path(
            row.get("mask_full_path"),
            EXTERNAL_VIEW_EXPORT_ROOT,
            row["dataset"],
            row["abs_path"],
            "mask_full",
        ),
        axis=1,
    )
    metadata["loose_path"] = metadata.apply(
        lambda row: resolve_external_view_path(
            row.get("loose_path"),
            EXTERNAL_VIEW_EXPORT_ROOT,
            row["dataset"],
            row["abs_path"],
            "loose",
        ),
        axis=1,
    )
metadata["mask_full_path"] = metadata["mask_full_path"].fillna(metadata["abs_path"])
metadata["loose_path"] = metadata["loose_path"].fillna(metadata["abs_path"])
metadata["mask_ok"] = metadata["mask_ok"].fillna(False).astype(bool)
display(metadata.groupby("dataset")["mask_ok"].mean())
        

In [ ]:
# =========================
# Training data, augmentation, sampler
# =========================

train_meta = metadata[(metadata["split"] == "train") & (metadata["has_identity"]) & (metadata["dataset"] != "TexasHornedLizards")].copy()
train_meta["label_text"] = train_meta["dataset"].astype(str) + "::" + train_meta["identity"].astype(str)
label_to_id = {label: idx for idx, label in enumerate(sorted(train_meta["label_text"].unique()))}
species_to_id = {species: idx for idx, species in enumerate(sorted(train_meta["dataset"].unique()))}
train_meta["label_id"] = train_meta["label_text"].map(label_to_id).astype(int)
train_meta["species_id"] = train_meta["dataset"].map(species_to_id).astype(int)

print("train images:", len(train_meta), "identities:", train_meta["label_id"].nunique())
display(train_meta.groupby("dataset")["label_id"].nunique())


class DegradationAugment:
    def __init__(self, p=0.75):
        self.p = p

    def blur(self, image):
        sigma = random.uniform(0.6, 1.8)
        return image.filter(ImageFilter.GaussianBlur(radius=sigma))

    def downsample(self, image):
        w, h = image.size
        factor = random.uniform(0.45, 0.85)
        small = image.resize((max(32, int(w * factor)), max(32, int(h * factor))), Image.Resampling.BILINEAR)
        return small.resize((w, h), Image.Resampling.BILINEAR)

    def jpeg(self, image):
        quality = random.randint(35, 80)
        buf = io.BytesIO()
        image.save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        with Image.open(buf) as tmp:
            return tmp.convert("RGB")

    def contrast(self, image):
        if random.random() < 0.5:
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.30))
        if random.random() < 0.5:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.78, 1.22))
        return image

    def noise(self, image):
        arr = np.asarray(image).astype(np.float32)
        noise = np.random.normal(0, random.uniform(3.0, 9.0), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(arr)

    def __call__(self, image):
        if random.random() > self.p:
            return image
        ops = [self.downsample, self.jpeg, self.contrast, self.noise]
        if random.random() < 0.55:
            ops.append(self.blur)
        random.shuffle(ops)
        image = ops[0](image)
        if random.random() < 0.35:
            image = ops[1](image)
        return image


random_erasing = T.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.3, 3.3), value="random")
train_degrade = DegradationAugment(p=0.80)
train_pre_tensor = T.Compose([
    T.RandomResizedCrop(TRAIN_IMAGE_SIZE, scale=(0.72, 1.0), ratio=(0.82, 1.18)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.03),
])
base_to_tensor = T.Compose([
    T.Resize((TRAIN_IMAGE_SIZE, TRAIN_IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class ARWildTrainDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.view_keys = ["abs_path", "mask_full_path", "loose_path"]
        self.view_weights = [TRAIN_VIEW_WEIGHTS["original"], TRAIN_VIEW_WEIGHTS["mask_full"], TRAIN_VIEW_WEIGHTS["loose"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        selected_key = random.choices(self.view_keys, weights=self.view_weights, k=1)[0]
        path = row[selected_key]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.open(row["abs_path"]).convert("RGB")
        image = train_degrade(image)
        image = train_pre_tensor(image)
        tensor = base_to_tensor(image)
        tensor = random_erasing(tensor)
        return {
            "image": tensor,
            "label": int(row["label_id"]),
            "species_id": int(row["species_id"]),
        }


class IdentityBatchSampler(Sampler):
    def __init__(self, labels, p=8, k=4, steps_per_epoch=200):
        self.labels = [int(x) for x in labels]
        self.p = int(p)
        self.k = int(k)
        self.steps_per_epoch = int(steps_per_epoch)
        self.label_to_indices = {}
        for idx, label in enumerate(self.labels):
            self.label_to_indices.setdefault(label, []).append(idx)
        self.unique_labels = list(self.label_to_indices.keys())

    def __iter__(self):
        for _ in range(self.steps_per_epoch):
            if len(self.unique_labels) >= self.p:
                chosen = random.sample(self.unique_labels, self.p)
            else:
                chosen = random.choices(self.unique_labels, k=self.p)
            batch = []
            for label in chosen:
                indices = self.label_to_indices[label]
                if len(indices) >= self.k:
                    batch.extend(random.sample(indices, self.k))
                else:
                    batch.extend(random.choices(indices, k=self.k))
            yield batch

    def __len__(self):
        return self.steps_per_epoch


train_dataset = ARWildTrainDataset(train_meta)
train_sampler = IdentityBatchSampler(train_meta["label_id"].tolist(), p=TRAIN_P, k=TRAIN_K, steps_per_epoch=STEPS_PER_EPOCH)
train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=TRAIN_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=PERSISTENT_WORKERS,
)
        

In [ ]:
# =========================
# ARWild model and training
# =========================

def gem_pool(x, p=3.0, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p).flatten(1)


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.30):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.s = s
        self.m = m
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, features, labels):
        cosine = F.linear(F.normalize(features), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - cosine.pow(2)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return logits * self.s


def batch_hard_triplet_loss(embeddings, labels, margin=0.30):
    if embeddings.size(0) < 3:
        return embeddings.new_tensor(0.0)
    dist = torch.cdist(embeddings, embeddings, p=2)
    labels = labels.view(-1)
    mask_pos = labels[:, None].eq(labels[None, :])
    mask_neg = ~mask_pos
    eye = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    mask_pos = mask_pos & ~eye
    hardest_pos = torch.where(mask_pos, dist, torch.full_like(dist, -1e9)).max(dim=1).values
    hardest_neg = torch.where(mask_neg, dist, torch.full_like(dist, 1e9)).min(dim=1).values
    valid = (hardest_pos > -1e8) & (hardest_neg < 1e8)
    if valid.sum() == 0:
        return embeddings.new_tensor(0.0)
    return F.relu(hardest_pos[valid] - hardest_neg[valid] + margin).mean()


class ARWildNet(nn.Module):
    def __init__(self, backbone_name, num_classes):
        super().__init__()
        import timm
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            num_classes=0,
            global_pool="",
        )
        self.num_features = self.backbone.num_features
        self.embed = nn.Linear(self.num_features, self.num_features)
        self.bnneck = nn.BatchNorm1d(self.num_features)
        self.bnneck.bias.requires_grad_(False)
        self.head = ArcMarginProduct(self.num_features, num_classes, s=ARC_SCALE, m=ARC_MARGIN)

    def extract(self, x):
        feats = self.backbone.forward_features(x)
        if feats.ndim == 4:
            feats = gem_pool(feats)
        x = self.embed(feats)
        x = self.bnneck(x)
        return x

    def forward(self, x, labels=None):
        feat = self.extract(x)
        emb = F.normalize(feat)
        if labels is None:
            return emb
        logits = self.head(emb, labels)
        return emb, logits


def build_arwild_model():
    model = ARWildNet(ARWILD_BACKBONE, num_classes=len(label_to_id)).to(DEVICE)
    if torch.cuda.is_available():
        try:
            model = model.to(memory_format=torch.channels_last)
        except Exception:
            pass
    return model


CHECKPOINT_PATH = CKPT_DIR / "arwildfusion_v3_best.pt"
LATEST_CHECKPOINT_PATH = CKPT_DIR / "arwildfusion_v3_latest.pt"


def find_latest_epoch_checkpoint():
    epoch_ckpts = sorted(CKPT_DIR.glob("arwild_epoch_*.pt"))
    best_epoch = -1
    best_path = None
    for path in epoch_ckpts:
        try:
            epoch = int(path.stem.split("_")[-1])
        except Exception:
            continue
        if epoch > best_epoch:
            best_epoch = epoch
            best_path = path
    return best_path, best_epoch


def build_train_state(model, optimizer, scheduler, scaler, epoch, epoch_loss, best_loss):
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch,
        "loss": epoch_loss,
        "best_loss": best_loss,
        "label_to_id": label_to_id,
        "species_to_id": species_to_id,
    }


resume_path, resume_epoch = find_latest_epoch_checkpoint()
start_epoch = 1
model = None

if CHECKPOINT_PATH.exists() and resume_epoch >= MAX_EPOCHS:
    print("[train] full training already completed:", CHECKPOINT_PATH)
else:
    model = build_arwild_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    best_loss = float("inf")

    if resume_path is not None:
        print("[train] resuming from:", resume_path)
        state = torch.load(resume_path, map_location="cpu")
        model.load_state_dict(state["model"], strict=True)
        if "optimizer" in state:
            optimizer.load_state_dict(state["optimizer"])
        if "scheduler" in state:
            scheduler.load_state_dict(state["scheduler"])
        else:
            for _ in range(int(state.get("epoch", 0))):
                scheduler.step()
        if "scaler" in state:
            scaler.load_state_dict(state["scaler"])
        start_epoch = int(state.get("epoch", 0)) + 1
        best_loss = float(state.get("best_loss", state.get("loss", float("inf"))))
        print(f"[train] start_epoch={start_epoch} best_loss={best_loss:.5f}")

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        running_ce = 0.0
        running_tri = 0.0
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f"train epoch {epoch}/{MAX_EPOCHS}")
        for step, batch in enumerate(pbar, start=1):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            images = images.contiguous(memory_format=torch.channels_last)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                emb, logits = model(images, labels)
                ce = criterion(logits, labels)
                tri = batch_hard_triplet_loss(emb, labels, margin=0.30)
                loss = ce + TRIPLET_WEIGHT * tri

            scaler.scale(loss / GRAD_ACCUM).backward()
            if step % GRAD_ACCUM == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += float(loss.item())
            running_ce += float(ce.item())
            running_tri += float(tri.item())
            pbar.set_postfix(loss=f"{running_loss / step:.4f}", ce=f"{running_ce / step:.4f}", tri=f"{running_tri / step:.4f}")

        if (step % GRAD_ACCUM) != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        epoch_loss = running_loss / max(step, 1)
        scheduler.step()
        print(f"[train] epoch={epoch} loss={epoch_loss:.5f}")

        state = build_train_state(model, optimizer, scheduler, scaler, epoch, epoch_loss, best_loss)
        torch.save(state, CKPT_DIR / f"arwild_epoch_{epoch}.pt")
        torch.save(state, LATEST_CHECKPOINT_PATH)
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            state["best_loss"] = best_loss
            torch.save(state, CHECKPOINT_PATH)
            print("[train] saved best checkpoint")

if model is not None:
    del model
    cleanup_memory("after train")

print("[train] using checkpoint:", CHECKPOINT_PATH)
        

In [ ]:
# =========================
# Feature extraction
# =========================

@dataclass
class ModelSpec:
    name: str
    batch_key: str
    transform: object
    build_fn: object


class PathImageDataset(Dataset):
    def __init__(self, paths, transform, tta_variant="identity"):
        self.paths = list(paths)
        self.transform = transform
        self.tta_variant = tta_variant

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (TRAIN_IMAGE_SIZE, TRAIN_IMAGE_SIZE), tuple(BACKGROUND_COLOR.tolist()))
        if self.tta_variant == "hflip":
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        image = self.transform(image)
        return image


def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 1:
        x = x[None, :]
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


def make_torch_transform(size, norm_mean, norm_std):
    return T.Compose([
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize(norm_mean, norm_std),
    ])


def build_arwild_infer_model():
    state = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model = build_arwild_model()
    model.load_state_dict(state["model"], strict=True)
    return model.eval()


class MegaWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        import timm
        self.model = timm.create_model("hf-hub:BVRA/MegaDescriptor-L-384", pretrained=True, num_classes=0)

    def forward(self, x):
        out = self.model(x)
        if isinstance(out, torch.Tensor):
            return out
        if isinstance(out, (tuple, list)):
            return out[0]
        raise TypeError(type(out))


def patch_transformers_for_miewid():
    try:
        from transformers.modeling_utils import PreTrainedModel
        current = getattr(PreTrainedModel, "all_tied_weights_keys", None)
        if current is None or not hasattr(current, "keys"):
            PreTrainedModel.all_tied_weights_keys = {}
    except Exception as e:
        print("[warn] Miew patch failed:", e)


class MiewWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        patch_transformers_for_miewid()
        from transformers import AutoModel
        self.model = AutoModel.from_pretrained("conservationxlabs/miewid-msv3", trust_remote_code=True)
        if not hasattr(self.model, "all_tied_weights_keys"):
            self.model.all_tied_weights_keys = {}

    def forward(self, x):
        out = self.model(x)
        if isinstance(out, torch.Tensor):
            return out
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, "last_hidden_state"):
            return out.last_hidden_state.mean(dim=1)
        if isinstance(out, dict):
            if "pooler_output" in out and out["pooler_output"] is not None:
                return out["pooler_output"]
            if "last_hidden_state" in out:
                return out["last_hidden_state"].mean(dim=1)
        if isinstance(out, (tuple, list)):
            return out[0]
        raise TypeError(type(out))


MODEL_SPECS = {
    "arwild": ModelSpec(
        name="arwild",
        batch_key="arwild",
        transform=make_torch_transform(TRAIN_IMAGE_SIZE, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        build_fn=build_arwild_infer_model,
    ),
    "mega": ModelSpec(
        name="mega",
        batch_key="mega",
        transform=make_torch_transform(384, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        build_fn=lambda: MegaWrapper().to(DEVICE).eval(),
    ),
    "miew": ModelSpec(
        name="miew",
        batch_key="miew",
        transform=make_torch_transform(440, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        build_fn=lambda: MiewWrapper().to(DEVICE).eval(),
    ),
}


def extract_embeddings(model_name, view_name, paths):
    spec = MODEL_SPECS[model_name]
    cache_key = hashlib.sha1(("|".join(paths[:50]) + f"::{len(paths)}::{model_name}::{view_name}").encode("utf-8")).hexdigest()[:14]
    cache_path = CACHE_DIR / f"{model_name}_{view_name}_{cache_key}.npy"
    if cache_path.exists():
        arr = np.load(cache_path)
        if arr.shape[0] == len(paths):
            print("[cache] loaded", cache_path.name, arr.shape)
            return l2_normalize(arr)

    print(f"[embed] loading {model_name} for {view_name}")
    cleanup_memory(f"before load {model_name}")
    model = spec.build_fn()
    if torch.cuda.is_available():
        try:
            model = model.to(memory_format=torch.channels_last)
        except Exception:
            pass

    all_variants = []
    for variant in TTA_VARIANTS:
        ds = PathImageDataset(paths, spec.transform, tta_variant=variant)
        bs = INFER_BATCH[spec.batch_key]
        while True:
            try:
                loader = DataLoader(
                    ds,
                    batch_size=bs,
                    shuffle=False,
                    num_workers=TRAIN_NUM_WORKERS,
                    pin_memory=torch.cuda.is_available(),
                    persistent_workers=PERSISTENT_WORKERS,
                )
                chunks = []
                with torch.inference_mode():
                    for batch in tqdm(loader, desc=f"{model_name}:{view_name}:{variant}:bs{bs}"):
                        batch = batch.contiguous(memory_format=torch.channels_last)
                        batch = batch.to(DEVICE, non_blocking=True)
                        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                            out = model(batch)
                        chunks.append(out.float().cpu().numpy())
                all_variants.append(l2_normalize(np.concatenate(chunks, axis=0)))
                break
            except RuntimeError as e:
                if "out of memory" not in str(e).lower() or bs == 1:
                    raise
                bs = max(1, bs // 2)
                cleanup_memory(f"oom {model_name} {view_name} bs->{bs}")

    arr = l2_normalize(np.mean(np.stack(all_variants, axis=0), axis=0))
    np.save(cache_path, arr.astype(np.float16))
    print("[cache] wrote", cache_path.name, arr.shape)

    del model
    cleanup_memory(f"after unload {model_name}")
    return arr


all_view_paths = {
    "original": metadata["abs_path"].tolist(),
    "mask_full": metadata["mask_full_path"].tolist(),
    "loose": metadata["loose_path"].tolist(),
}

feature_bank = {}
for model_name in ["arwild", "mega", "miew"]:
    for view_name in ["original", "mask_full", "loose"]:
        feature_bank[(model_name, view_name)] = extract_embeddings(model_name, view_name, all_view_paths[view_name])
        cleanup_memory(f"done {model_name} {view_name}")
        

In [ ]:
# =========================
# Clustering and submission
# =========================

def query_expansion(features, k=5, alpha=0.5):
    if len(features) <= 1:
        return features
    k = max(1, min(int(k), len(features)))
    sim = np.dot(features, features.T)
    knn = np.argsort(sim, axis=1)[:, -k:]
    expanded = np.zeros_like(features)
    for i in range(features.shape[0]):
        expanded[i] = features[i] + alpha * np.mean(features[knn[i]], axis=0)
    return normalize(expanded, axis=1, norm="l2")


def run_agglomerative(features, threshold):
    if len(features) == 0:
        return np.array([], dtype=int)
    if len(features) == 1:
        return np.array([0], dtype=int)
    dist = np.clip(1.0 - np.dot(features, features.T), 0.0, 2.0)
    try:
        model = AgglomerativeClustering(
            n_clusters=None,
            metric="precomputed",
            linkage="average",
            distance_threshold=float(threshold),
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=None,
            affinity="precomputed",
            linkage="average",
            distance_threshold=float(threshold),
        )
    return model.fit_predict(dist)


def relabel_dense(labels):
    mapping = {label: idx for idx, label in enumerate(pd.unique(labels))}
    return np.array([mapping[x] for x in labels], dtype=int)


def refine_large_clusters(features, labels, split_cap, split_threshold, internal_sim_floor):
    labels = np.asarray(labels).copy()
    if len(labels) <= 1:
        return labels
    sim = np.dot(features, features.T)
    next_label = int(labels.max()) + 1
    for label in sorted(pd.unique(labels)):
        idx = np.where(labels == label)[0]
        if len(idx) <= split_cap:
            continue
        sub = sim[np.ix_(idx, idx)]
        tri = sub[np.triu_indices(len(idx), 1)]
        mean_sim = float(tri.mean()) if len(tri) else 1.0
        if mean_sim >= internal_sim_floor:
            continue
        sub_labels = run_agglomerative(features[idx], split_threshold)
        sub_labels = relabel_dense(sub_labels)
        if len(np.unique(sub_labels)) == 1:
            continue
        for local in np.unique(sub_labels):
            local_idx = idx[sub_labels == local]
            labels[local_idx] = next_label
            next_label += 1
    return relabel_dense(labels)


def build_candidate_features(species_rows, species):
    cfg = SPECIES_CONFIG[species]
    dominant = cfg["dominant_fixed"]
    support = cfg["support_fixed"]
    candidate_features = []
    for view_preset_name, model_preset_name in cfg["candidate_order"]:
        view_weights = VIEW_WEIGHT_PRESETS[view_preset_name]
        model_weights = MODEL_WEIGHT_PRESETS[model_preset_name]
        parts = []
        for model_name, role in [("arwild", "trained"), (dominant, "dominant"), (support, "support")]:
            role_weight = model_weights[role]
            for view_name in ["original", "mask_full", "loose"]:
                view_weight = view_weights[view_name]
                if role_weight <= 0 or view_weight <= 0:
                    continue
                arr = feature_bank[(model_name, view_name)][species_rows["row_idx"].tolist()]
                parts.append(role_weight * view_weight * arr)
        fused = l2_normalize(np.concatenate(parts, axis=1))
        candidate_features.append({
            "view_preset": view_preset_name,
            "model_preset": model_preset_name,
            "features": fused,
        })
    return candidate_features


all_parts = []
report_rows = []
candidate_rows = []

for species in metadata["dataset"].unique():
    print("\n" + "=" * 90)
    print("species:", species)
    cfg = SPECIES_CONFIG[species]
    species_rows = metadata[metadata["dataset"] == species].copy().reset_index(drop=True)
    candidate_banks = build_candidate_features(species_rows, species)

    train_rows = species_rows[(species_rows["split"] == "train") & (species_rows["has_identity"])].copy()
    test_rows = species_rows[species_rows["split"] == "test"].copy()
    row_to_local = {int(row_idx): i for i, row_idx in enumerate(species_rows["row_idx"].tolist())}
    train_idx = [row_to_local[int(x)] for x in train_rows["row_idx"].tolist()]
    test_idx = [row_to_local[int(x)] for x in test_rows["row_idx"].tolist()]

    best_score = -1e9
    best_meta = None
    best_test_features = None

    for bank_id, bank in enumerate(candidate_banks):
        for qe_k in cfg["qe_grid"]:
            for thr in cfg["threshold_grid"]:
                train_score = np.nan
                if len(train_idx) > 2 and train_rows["identity"].nunique() > 1:
                    train_features = bank["features"][train_idx]
                    train_features = query_expansion(train_features, k=qe_k)
                    pred = run_agglomerative(train_features, thr)
                    pred = refine_large_clusters(train_features, pred, cfg["split_cap"], cfg["split_threshold"], cfg["split_internal_sim"])
                    train_score = float(adjusted_rand_score(train_rows["identity"].astype(str).values, pred))
                else:
                    if species == "TexasHornedLizards":
                        train_score = 0.0

                candidate_rows.append({
                    "species": species,
                    "bank_id": bank_id,
                    "view_preset": bank["view_preset"],
                    "model_preset": bank["model_preset"],
                    "qe_k": qe_k,
                    "threshold": thr,
                    "train_ari": train_score,
                })

                score = -1.0 if np.isnan(train_score) else train_score
                if species == "TexasHornedLizards":
                    bonus = 0.05 if bank["model_preset"] == "trained_dominant" else 0.0
                    score = bonus - abs(thr - cfg["fallback_threshold"])

                if score > best_score:
                    best_score = score
                    best_meta = {
                        "view_preset": bank["view_preset"],
                        "model_preset": bank["model_preset"],
                        "qe_k": qe_k,
                        "threshold": float(thr),
                        "train_ari": None if np.isnan(train_score) else float(train_score),
                    }
                    best_test_features = query_expansion(bank["features"][test_idx], k=qe_k)

    labels = run_agglomerative(best_test_features, best_meta["threshold"])
    labels = refine_large_clusters(best_test_features, labels, cfg["split_cap"], cfg["split_threshold"], cfg["split_internal_sim"])
    labels = relabel_dense(labels)

    part = pd.DataFrame({
        "image_id": test_rows["image_id"].values,
        "cluster": [f"cluster_{species}_{x}" for x in labels],
    })
    all_parts.append(part)
    vc = part["cluster"].value_counts()
    report_rows.append({
        "species": species,
        "n_test_images": int(len(test_rows)),
        "n_clusters": int(part["cluster"].nunique()),
        "max_cluster_size": int(vc.max()) if len(vc) else 0,
        "singletons": int((vc == 1).sum()) if len(vc) else 0,
        "mask_coverage": float(species_rows["mask_ok"].mean()),
        "best_view_preset": best_meta["view_preset"],
        "best_model_preset": best_meta["model_preset"],
        "best_qe_k": best_meta["qe_k"],
        "best_threshold": best_meta["threshold"],
        "best_train_ari": best_meta["train_ari"] if best_meta["train_ari"] is not None else "N/A",
    })
    print("report:", report_rows[-1])

predictions = pd.concat(all_parts, ignore_index=True)
submission = sample_submission[["image_id"]].merge(predictions, on="image_id", how="left")
assert len(submission) == len(sample_submission)
assert submission["cluster"].notna().all()

submission_path = WORK_DIR / "submission.csv"
versioned_submission_path = SUB_DIR / f"submission_{VERSION}.csv"
report_path = WORK_DIR / "arwildfusion_v3_report.csv"
candidate_path = WORK_DIR / "arwildfusion_v3_candidates.csv"
submission.to_csv(submission_path, index=False)
submission.to_csv(versioned_submission_path, index=False)
pd.DataFrame(report_rows).to_csv(report_path, index=False)
pd.DataFrame(candidate_rows).to_csv(candidate_path, index=False)

print("\nWrote:", submission_path)
print("Wrote:", versioned_submission_path)
print("Wrote:", report_path)
print("Wrote:", candidate_path)
display(pd.DataFrame(report_rows))
display(submission.head())
        

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
for candidate in [repo, *repo.parents]:
    if (candidate / "paper_submissions_manifest.csv").exists():
        repo = candidate
        break
sys.path.insert(0, str(repo / "src"))

from notebook_tools import verify_submission
verify_submission(Path.cwd() / "submission.csv", "shape_constrained_fusion_partition", repo)